# CA3 HW3: Energy-Based and Score-Based Models on MNIST

**Objectives**

- Implement EBM with Langevin sampling and contrastive divergence.
- Implement NCSN with weighted DSM and annealed Langevin dynamics (unconditional + conditional).
- Provide training, sampling, and denoising pipelines with reproducibility hooks.

**Structure**

1. Setup and configuration
2. Data loading and visualization
3. EBM model, training, sampling, denoising
4. NCSN model, training, sampling (ALD), denoising, conditional variant
5. Results logging placeholders
6. Reproducibility notes


In [ ]:
# Setup and Configuration
import os, random
import numpy as np
import torch
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
DATA_ROOT = PROJECT_ROOT / "data" / "mnist"


def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# Enable relative imports in notebook
import sys
sys.path.insert(0, str(Path.cwd()))

In [ ]:
# Install requirements (run this in Colab)
!pip install -r ../requirements.txt

In [ ]:
# Configs
from config import DataConfig, EBMConfig, NCSNConfig, RunPaths

data_cfg = DataConfig()
ebm_cfg = EBMConfig(device=device)
ncsn_cfg = NCSNConfig(device=device)
paths = RunPaths()
paths.ensure()
data_cfg, ebm_cfg, ncsn_cfg

In [ ]:
# Data loading and a quick peek
from data import mnist_dataloaders
from torchvision.utils import make_grid, save_image
import matplotlib.pyplot as plt

train_loader, test_loader = mnist_dataloaders(data_cfg, normalize_to_minus1_1=False)
images, labels = next(iter(train_loader))
grid = make_grid(images[:16], nrow=4)
save_image(grid, paths.images / "mnist_sample.png")
plt.figure(figsize=(4, 4))
plt.axis("off")
plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
plt.show()

In [ ]:
# EBM model, sampler, and a short training utility (configurable)
from ebm_model import ConvEnergyModel
from ebm_sampling import LangevinSampler, sample_from_noise
from torch import optim
from tqdm import tqdm
from torchvision.utils import save_image


def train_ebm(train_loader, test_loader, cfg: EBMConfig, epochs: int = 1):
    model = ConvEnergyModel().to(cfg.device)
    sampler = LangevinSampler(model, cfg)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    history = []
    for epoch in range(1, epochs + 1):
        loop = tqdm(train_loader, desc=f"EBM epoch {epoch}/{epochs}", leave=False)
        for x, _ in loop:
            x = x.to(cfg.device)
            x_fake = sampler(torch.rand_like(x))
            e_real = model(x)
            e_fake = model(x_fake)
            data_term = e_real.mean() - e_fake.detach().mean()
            reg_term = cfg.lambda_reg * (e_real.pow(2).mean() + e_fake.detach().pow(2).mean())
            loss = data_term + reg_term
            opt.zero_grad()
            loss.backward()
            opt.step()
            history.append(loss.item())
        samples = sample_from_noise(model, cfg, (16, 1, 28, 28))
        grid = make_grid(samples, nrow=4, normalize=True)
        save_image(grid, paths.images / "ebm_demo_samples.png")
        plt.figure(figsize=(4, 4))
        plt.axis("off")
        plt.imshow(grid.permute(1, 2, 0))
        plt.show()
    return model, history


# Note: For full training, run ebm_train.py from the module; this cell is for interactive/smoke usage.

In [ ]:
# Full EBM Training Pipeline
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any
from torch import optim
from tqdm import tqdm
from config import DataConfig, EBMConfig, RunPaths
from data import mnist_dataloaders
from ebm_model import ConvEnergyModel
from ebm_sampling import LangevinSampler, sample_from_noise
from utils import save_grid, set_seed, ensure_dir, write_run_info


def full_train_ebm(cfg_data: DataConfig, cfg_model: EBMConfig, output_dir: Path) -> Dict[str, Any]:
    set_seed(cfg_data.seed)
    train_loader, test_loader = mnist_dataloaders(cfg_data)
    device = cfg_model.device

    model = ConvEnergyModel().to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg_model.lr)
    sampler = LangevinSampler(model, cfg_model)

    history = {"loss": [], "E_real": [], "E_fake": []}
    ensure_dir(output_dir)
    checkpoint_path = output_dir / "ebm_ckpt.pt"
    write_run_info(
        output_dir,
        configs={"data": asdict(cfg_data), "model": asdict(cfg_model)},
        notes={"script": "notebook full_train_ebm"},
        device=str(device),
    )

    for epoch in range(1, cfg_model.epochs + 1):
        progress = tqdm(
            train_loader, desc=f"EBM Epoch {epoch}/{cfg_model.epochs}", leave=False
        )
        for step, (x_real, _) in enumerate(progress, start=1):
            x_real = x_real.to(device)
            x_fake = sampler(torch.rand_like(x_real))

            E_real = model(x_real)
            E_fake = model(x_fake)

            data_term = E_real.mean() - E_fake.detach().mean()
            reg_term = cfg_model.lambda_reg * (
                E_real.pow(2).mean() + E_fake.detach().pow(2).mean()
            )
            loss = data_term + reg_term

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            history["loss"].append(loss.item())
            history["E_real"].append(E_real.mean().item())
            history["E_fake"].append(E_fake.mean().item())

            if step % cfg_model.log_interval == 0:
                progress.set_postfix(loss=f"{loss.item():.3f}")

        # Save training samples each epoch
        with torch.no_grad():
            samples = sample_from_noise(
                model, cfg_model, (cfg_model.sample_grid, 1, 28, 28)
            )
            save_grid(samples, output_dir / f"ebm_samples_epoch{epoch}.png", nrow=4)

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            },
            checkpoint_path,
        )

    # Save simple visualizations of loss and energies
    try:
        import matplotlib.pyplot as plt
        plt.figure(figsize=(6, 4))
        plt.plot(history["loss"], label="loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("EBM Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ebm_loss.png")
        plt.close()

        plt.figure(figsize=(6, 4))
        plt.plot(history["E_real"], label="E_real")
        plt.plot(history["E_fake"], label="E_fake")
        plt.xlabel("Step")
        plt.ylabel("Energy")
        plt.title("EBM Energies")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ebm_energy.png")
        plt.close()
    except Exception:
        # Best-effort visualization; continue even if matplotlib is unavailable.
        pass

    return history


# Example: full_ebm_cfg = EBMConfig(epochs=10)
# full_train_ebm(DataConfig(), full_ebm_cfg, paths.images / "ebm")

In [ ]:
# NCSN model, DSM loss, and ALD sampling utilities
from ncsn_model import ScoreNet
from ncsn_loss import dsm_loss
from ncsn_sampling import sample as ncsn_sample
from torchvision.utils import save_image


def train_ncsn(
    train_loader, cfg: NCSNConfig, epochs: int = 1, conditional: bool = False
):
    cfg.conditional = conditional
    model = ScoreNet(cfg).to(cfg.device)
    opt = optim.Adam(model.parameters(), lr=cfg.lr)
    sigmas = cfg.sigmas
    history = []
    loop_epochs = range(1, epochs + 1)
    for epoch in loop_epochs:
        loop = tqdm(train_loader, desc=f"NCSN epoch {epoch}/{epochs}", leave=False)
        for x, y in loop:
            x = x.to(cfg.device) * 2 - 1
            y_lbl = y.to(cfg.device) if conditional else None
            loss = dsm_loss(model, x, cfg, sigmas, y_lbl)
            opt.zero_grad()
            loss.backward()
            opt.step()
            history.append(loss.item())
        with torch.no_grad():
            y_samples = (
                (torch.arange(0, 16, device=cfg.device) % cfg.num_classes)
                if conditional
                else None
            )
            samples = ncsn_sample(model, cfg, num_samples=16, y=y_samples)
            grid = make_grid(
                (samples + 1) / 2, nrow=4, normalize=True, value_range=(0, 1)
            )
            save_image(grid, paths.images / "ncsn_demo_samples.png")
            plt.figure(figsize=(4, 4))
            plt.axis("off")
            plt.imshow(grid.permute(1, 2, 0))
            plt.show()
    return model, history


# Note: For full training, use ncsn_train.py entrypoint. This cell is for interactive smoke tests or short runs.

In [ ]:
# Full NCSN Training Pipeline
import matplotlib.pyplot as plt
from dataclasses import asdict
from pathlib import Path
from typing import Dict, Any, Optional

import torch
from torch import optim
from tqdm import tqdm

from config import NCSNConfig, DataConfig, RunPaths
from data import mnist_dataloaders
from ncsn_model import ScoreNet
from ncsn_loss import dsm_loss
from ncsn_sampling import sample
from utils import save_grid, set_seed, ensure_dir, write_run_info


def full_train_ncsn(cfg: NCSNConfig, output_dir: Path, conditional: bool = False) -> Dict[str, Any]:
    cfg.conditional = conditional
    set_seed(42)

    data_cfg = DataConfig(
        batch_size=cfg.batch_size, num_workers=cfg.num_workers, channels=cfg.channels
    )
    train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=True)

    device = cfg.device
    model = ScoreNet(cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)
    sigmas = cfg.sigmas

    ensure_dir(output_dir)
    history = {"loss": []}
    checkpoint_path = output_dir / ("ncsn_cond.pt" if conditional else "ncsn.pt")
    write_run_info(
        output_dir,
        configs={
            "data": asdict(data_cfg),
            "model": asdict(cfg),
            "conditional": {"enabled": conditional},
        },
        notes={"script": "notebook full_train_ncsn"},
        device=str(device),
    )

    for epoch in range(1, cfg.epochs + 1):
        progress = tqdm(
            train_loader, desc=f"NCSN Epoch {epoch}/{cfg.epochs}", leave=False
        )
        for x, labels in progress:
            x = x.to(device)
            x = x * 2 - 1  # map to [-1, 1]
            y = labels.to(device) if conditional else None

            loss = dsm_loss(model, x, cfg, sigmas, y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            history["loss"].append(loss.item())
            progress.set_postfix(loss=f"{loss.item():.3f}")

        with torch.no_grad():
            y_samples: Optional[torch.Tensor] = None
            if conditional:
                y_samples = torch.arange(0, 16, device=device) % cfg.num_classes
            samples = sample(model, cfg, num_samples=16, y=y_samples)
            samples = (samples + 1) / 2.0
            save_grid(samples, output_dir / f"samples_epoch{epoch}.png", nrow=4)

        torch.save(
            {
                "model": model.state_dict(),
                "optimizer": optimizer.state_dict(),
                "epoch": epoch,
            },
            checkpoint_path,
        )

    # Save loss visualization
    try:
        plt.figure(figsize=(6, 4))
        plt.plot(history["loss"], label="DSM loss")
        plt.xlabel("Step")
        plt.ylabel("Loss")
        plt.title("NCSN DSM Loss")
        plt.legend()
        plt.tight_layout()
        plt.savefig(output_dir / "ncsn_loss.png")
        plt.close()
    except Exception:
        # Best-effort plotting
        pass

    return history


# Example: full_ncsn_cfg = NCSNConfig(epochs=30)
# full_train_ncsn(full_ncsn_cfg, paths.images / "ncsn", conditional=False)
# full_train_ncsn(full_ncsn_cfg, paths.images / "ncsn_cond", conditional=True)

## Usage Notes

- Full training pipelines are available in the notebook: `full_train_ebm` and `full_train_ncsn`.
- Inference pipelines: `ebm_generate_and_denoise` and `ncsn_generate_and_denoise`.
- For quick demos, use the short training functions with `epochs=1`.
- Ensure `torch` and `torchvision` are installed (see `requirements.txt`).
- Figures are saved to `paths.images` subdirectories; inline plots are for demos.

In [ ]:
# EBM Inference Pipeline: Generation and Denoising
from pathlib import Path
import torch

from config import DataConfig, EBMConfig, RunPaths
from data import mnist_dataloaders
from ebm_model import ConvEnergyModel
from ebm_sampling import sample_from_noise, LangevinSampler
from utils import save_grid, set_seed, ensure_dir


def load_ebm_model(checkpoint: Path, device: torch.device) -> ConvEnergyModel:
    model = ConvEnergyModel().to(device)
    state = torch.load(checkpoint, map_location=device)
    model.load_state_dict(state["model"])
    model.eval()
    return model


def ebm_generate_and_denoise(
    checkpoint: Path, output_dir: Path, data_cfg: DataConfig, ebm_cfg: EBMConfig
) -> None:
    set_seed(data_cfg.seed)
    train_loader, _ = mnist_dataloaders(data_cfg)
    ensure_dir(output_dir)
    model = load_ebm_model(checkpoint, ebm_cfg.device)
    sampler = LangevinSampler(model, ebm_cfg)

    with torch.no_grad():
        samples = sample_from_noise(model, ebm_cfg, (16, 1, 28, 28))
        save_grid(samples, output_dir / "ebm_samples_final.png", nrow=4)

    # Denoise a few training digits
    x_real, _ = next(iter(train_loader))
    x_real = x_real[:16].to(ebm_cfg.device)
    noise = torch.randn_like(x_real) * 0.3
    noisy = (x_real + noise).clamp(0.0, 1.0)
    denoised = sampler(noisy)
    save_grid(x_real, output_dir / "ebm_real.png", nrow=4)
    save_grid(noisy, output_dir / "ebm_noisy.png", nrow=4)
    save_grid(denoised, output_dir / "ebm_denoised.png", nrow=4)


# Example: ebm_generate_and_denoise(paths.images / "ebm" / "ebm_ckpt.pt", paths.images / "ebm_infer", DataConfig(), EBMConfig())

In [ ]:
# NCSN Inference Pipeline: Sampling and Denoising
from pathlib import Path
from typing import Optional, Sequence
import torch

from config import NCSNConfig, DataConfig, RunPaths
from data import mnist_dataloaders
from ncsn_model import ScoreNet
from ncsn_sampling import sample, annealed_langevin_dynamics
from utils import save_grid, ensure_dir


def load_ncsn_model(
    checkpoint: Path, cfg: NCSNConfig, conditional: bool = False
) -> ScoreNet:
    cfg.conditional = conditional
    model = ScoreNet(cfg).to(cfg.device)
    state = torch.load(checkpoint, map_location=cfg.device)
    model.load_state_dict(state["model"])
    model.eval()
    return model


@torch.no_grad()
def ncsn_generate_and_denoise(
    checkpoint: Path,
    output_dir: Path,
    cfg: NCSNConfig,
    conditional: bool = False,
    noise_levels: Sequence[float] = (0.2, 0.4, 0.6),
) -> None:
    ensure_dir(output_dir)
    data_cfg = DataConfig(batch_size=16)
    train_loader, _ = mnist_dataloaders(data_cfg, normalize_to_minus1_1=True)
    model = load_ncsn_model(checkpoint, cfg, conditional)

    y_samples: Optional[torch.Tensor] = None
    if conditional:
        y_samples = torch.arange(0, 16, device=cfg.device) % cfg.num_classes
    samples = sample(model, cfg, num_samples=16, y=y_samples)
    save_grid((samples + 1) / 2.0, output_dir / "ncsn_samples.png", nrow=4)

    x_real, labels = next(iter(train_loader))
    x_real = x_real.to(cfg.device)[:16] * 2 - 1
    y = labels.to(cfg.device)[:16] if conditional else None

    for nl in noise_levels:
        noisy = x_real + nl * torch.randn_like(x_real)
        sigmas = torch.tensor([nl], device=cfg.device)
        denoised = annealed_langevin_dynamics(model, cfg, sigmas, noisy.clone(), y)
        save_grid((noisy + 1) / 2.0, output_dir / f"noisy_{nl:.2f}.png", nrow=4)
        save_grid((denoised + 1) / 2.0, output_dir / f"denoised_{nl:.2f}.png", nrow=4)


# Example: ncsn_generate_and_denoise(paths.images / "ncsn" / "ncsn.pt", paths.images / "ncsn_infer", NCSNConfig(), conditional=False)

## Quick demo run (short, inline)

- Runs 1 epoch EBM and NCSN (unconditional) with default configs.
- Uses small epochs to keep runtime manageable; for full quality use the scripts.
- Displays sample grids inline (not saved); ensure `torch` is installed and GPU is recommended.


In [ ]:
# Demo: run short EBM and NCSN training and show samples
# WARNING: still compute-intensive; adjust epochs/steps if needed.

# EBM short run (1 epoch)
short_ebm_cfg = ebm_cfg
short_ebm_cfg.langevin_steps = 20  # faster demo
short_ebm_cfg.sample_grid = 8
model_ebm, ebm_hist = train_ebm(train_loader, test_loader, short_ebm_cfg, epochs=1)

# NCSN short run (1 epoch, unconditional)
short_ncsn_cfg = ncsn_cfg
short_ncsn_cfg.num_levels = 5  # faster demo
short_ncsn_cfg.langevin_steps = 50
model_ncsn, ncsn_hist = train_ncsn(
    train_loader, short_ncsn_cfg, epochs=1, conditional=False
)

print(f"EBM steps: {len(ebm_hist)} losses logged")
print(f"NCSN steps: {len(ncsn_hist)} losses logged")

## Visualize demo losses

Plot the loss histories from the short demo runs to quickly inspect optimization behavior.


In [ ]:
import matplotlib.pyplot as plt

if "ebm_hist" in locals() and ebm_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ebm_hist)
    plt.title("EBM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.savefig(paths.images / "ebm_demo_loss.png")
    plt.show()
else:
    print("Run the demo cell to populate ebm_hist.")

if "ncsn_hist" in locals() and ncsn_hist:
    plt.figure(figsize=(6, 4))
    plt.plot(ncsn_hist)
    plt.title("NCSN DSM Loss (demo)")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.savefig(paths.images / "ncsn_demo_loss.png")
    plt.show()
else:
    print("Run the demo cell to populate ncsn_hist.")

## Display saved figures if available

Checks for images produced by the script entrypoints (e.g., `images/ebm/ebm_samples_epoch10.png`) and shows them inline when present.


In [ ]:
from matplotlib import image as mpimg

candidates = [
    paths.images / "ebm" / "ebm_samples_epoch10.png",
    paths.images / "ebm" / "ebm_denoised_epoch10.png",
    paths.images / "ncsn" / "samples_epoch30.png",
    paths.images / "ncsn_cond" / "samples_epoch30.png",
    paths.images / "ncsn_infer" / "denoised_0.40.png",
]

for img_path in candidates:
    if img_path.exists():
        img = mpimg.imread(img_path)
        plt.figure(figsize=(5, 5))
        plt.axis("off")
        plt.title(img_path.name)
        plt.imshow(img)
        plt.show()
    else:
        print(f"Missing: {img_path}")